In [ ]:
!pip install tensorflow
!pip install sktime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 18.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# HITSIGHT / vHIT IMPULSE CLASSIFICATION
# CNN #4 — TEMPORAL CONVOLUTIONAL NETWORK (TCN)
# ============================================================

import os
import copy
import random
import numpy as np
import pandas as pd

import torch
import torch._utils  # Prevents internal PyTorch C-extension binding errors
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# PATHS (SAVING TO GOOGLE DRIVE)
# ============================================================

X_PATH = "/content/drive/MyDrive/_left_ready.csv"
Y_PATH = "/content/drive/MyDrive/_labels_ready_new.csv"

# Updated to save directly inside Google Drive
SAVE_DIR = "/content/drive/MyDrive/hitsight_models"
os.makedirs(SAVE_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(
    SAVE_DIR,
    "cnn4_tcn_best.pt"
)

# ============================================================
# LOAD DATA
# ============================================================

X_df = pd.read_csv(X_PATH)
y_df = pd.read_csv(Y_PATH)

print("Raw X shape:", X_df.shape)
print("Raw Y shape:", y_df.shape)

# Remove accidental index columns
X_df = X_df.loc[
    :,
    ~X_df.columns.astype(str).str.startswith("Unnamed")
]

# ============================================================
# LABELS
# ============================================================

y_raw = y_df["Labels"].astype(str).values

print("Number of labels:", len(y_raw))
print("Number of signal rows:", len(X_df))

# ============================================================
# VERIFY 2:1 STRUCTURE
# ============================================================

if len(X_df) != 2 * len(y_raw):
    raise ValueError(
        f"Expected exactly 2 signal rows per label, "
        f"but found {len(X_df)} signal rows and "
        f"{len(y_raw)} labels."
    )

# ============================================================
# CONVERT SIGNAL DATA
# ============================================================

X_raw = X_df.astype(np.float32).values

n_impulses = len(y_raw)
n_samples = X_raw.shape[1]

print("Samples per signal:", n_samples)

# ============================================================
# RESHAPE
# ============================================================

X = X_raw.reshape(
    n_impulses,
    2,
    n_samples
)

y = y_raw

print("\nFull dataset:")
print("X:", X.shape)
print("y:", y.shape)

# ============================================================
# EXTRACT 25–75 ms WINDOW
# ============================================================

X = X[:, :, 25:75]

print("\nAfter 25–75 ms extraction:")
print("X:", X.shape)

# ============================================================
# LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("\nClasses:")

for i, label in enumerate(label_encoder.classes_):
    print(i, "=", label)

num_classes = len(label_encoder.classes_)

print("\nNumber of classes:", num_classes)

# ============================================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.30,
    random_state=SEED,
    stratify=y_encoded
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("\nSplits:")
print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)

# ============================================================
# NORMALIZATION
# ============================================================

mean = X_train.mean(
    axis=(0, 2),
    keepdims=True
)

std = X_train.std(
    axis=(0, 2),
    keepdims=True
)

std[std < 1e-6] = 1.0

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

# ============================================================
# DATASET
# ============================================================

class VHITDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )
        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = VHITDataset(
    X_train,
    y_train
)

val_dataset = VHITDataset(
    X_val,
    y_val
)

test_dataset = VHITDataset(
    X_test,
    y_test
)

# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

# ============================================================
# CAUSAL DILATED CONVOLUTION
# ============================================================

class Chomp1d(nn.Module):

    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size]


# ============================================================
# TCN RESIDUAL BLOCK
# ============================================================

class TemporalBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        dilation,
        dropout
    ):
        super().__init__()

        padding = (
            kernel_size - 1
        ) * dilation

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.chomp1 = Chomp1d(
            padding
        )

        self.bn1 = nn.BatchNorm1d(
            out_channels
        )

        self.relu1 = nn.ReLU(
            inplace=True
        )

        self.dropout1 = nn.Dropout(
            dropout
        )

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.chomp2 = Chomp1d(
            padding
        )

        self.bn2 = nn.BatchNorm1d(
            out_channels
        )

        self.relu2 = nn.ReLU(
            inplace=True
        )

        self.dropout2 = nn.Dropout(
            dropout
        )

        # Residual projection
        if in_channels != out_channels:
            self.downsample = nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=1
            )
        else:
            self.downsample = nn.Identity()

        self.final_relu = nn.ReLU(
            inplace=True
        )

        self._initialize_weights()

    def _initialize_weights(self):
        nn.init.kaiming_normal_(
            self.conv1.weight,
            nonlinearity="relu"
        )
        nn.init.kaiming_normal_(
            self.conv2.weight,
            nonlinearity="relu"
        )
        if isinstance(
            self.downsample,
            nn.Conv1d
        ):
            nn.init.kaiming_normal_(
                self.downsample.weight,
                nonlinearity="relu"
            )

    def forward(self, x):
        residual = self.downsample(x)

        out = self.conv1(x)
        out = self.chomp1(out)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.dropout1(out)

        out = self.conv2(out)
        out = self.chomp2(out)
        out = self.bn2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        out = out + residual

        return self.final_relu(out)


# ============================================================
# CNN #4 — TEMPORAL CONVOLUTIONAL NETWORK
# ============================================================

class TCN1D(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.network = nn.Sequential(
            TemporalBlock(
                in_channels=2,
                out_channels=32,
                kernel_size=3,
                dilation=1,
                dropout=0.10
            ),
            TemporalBlock(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                dilation=2,
                dropout=0.10
            ),
            TemporalBlock(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                dilation=4,
                dropout=0.15
            ),
            TemporalBlock(
                in_channels=128,
                out_channels=128,
                kernel_size=3,
                dilation=8,
                dropout=0.15
            ),
            TemporalBlock(
                in_channels=128,
                out_channels=256,
                kernel_size=3,
                dilation=4,
                dropout=0.15
            )
        )

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(
                256,
                128
            ),
            nn.ReLU(
                inplace=True
            ),
            nn.Dropout(
                0.40
            ),
            nn.Linear(
                128,
                num_classes
            )
        )

    def forward(self, x):
        x = self.network(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:", device)

# ============================================================
# MODEL
# ============================================================

model = TCN1D(
    num_classes=num_classes
).to(device)

print("\nModel:")
print(model)

# ============================================================
# PARAMETER COUNT
# ============================================================

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"\nTrainable parameters: "
    f"{trainable_params:,}"
)

# ============================================================
# LOSS & OPTIMIZER
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# ============================================================
# LR SCHEDULER
# ============================================================

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=5
)

# ============================================================
# TRAINING SETTINGS
# ============================================================

EPOCHS = 100
PATIENCE = 15

best_val_acc = -1.0
best_val_f1 = -1.0

epochs_without_improvement = 0

# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(EPOCHS):

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        train_loss += loss.item()

        predictions = logits.argmax(dim=1)

        train_correct += (
            predictions == y_batch
        ).sum().item()

        train_total += y_batch.size(0)

    train_acc = train_correct / train_total
    avg_train_loss = train_loss / len(train_loader)

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_predictions = []
    val_targets = []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)

            logits = model(X_batch)

            predictions = (
                logits
                .argmax(dim=1)
                .cpu()
                .numpy()
            )

            val_predictions.extend(predictions)
            val_targets.extend(y_batch.numpy())

    val_predictions = np.array(val_predictions)
    val_targets = np.array(val_targets)

    val_acc = accuracy_score(
        val_targets,
        val_predictions
    )

    val_macro_f1 = f1_score(
        val_targets,
        val_predictions,
        average="macro",
        zero_division=0
    )

    scheduler.step(val_acc)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch+1:03d}/{EPOCHS} | "
        f"Loss: {avg_train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Macro-F1: {val_macro_f1:.4f} | "
        f"LR: {current_lr:.2e}"
    )

    # ========================================================
    # BEST MODEL (SAVES DIRECTLY TO GOOGLE DRIVE)
    # ========================================================

    is_better = (
        val_macro_f1 > best_val_f1
        or (
            val_macro_f1 == best_val_f1
            and val_acc > best_val_acc
        )
    )

    if is_better:
        best_val_f1 = val_macro_f1
        best_val_acc = val_acc

        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "mean": mean,
                "std": std,
                "classes": label_encoder.classes_,
                "val_accuracy": val_acc,
                "val_macro_f1": val_macro_f1,
                "epoch": epoch + 1
            },
            BEST_MODEL_PATH
        )

        print(
            f">>> NEW BEST MODEL SAVED TO DRIVE "
            f"(Val Acc = {val_acc:.4f}, "
            f"Val Macro-F1 = {val_macro_f1:.4f})"
        )

    else:
        epochs_without_improvement += 1

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if epochs_without_improvement >= PATIENCE:
        print(
            f"\nEarly stopping at epoch {epoch+1}."
        )
        break


# ============================================================
# LOAD BEST MODEL FROM DRIVE
# ============================================================

print("\nLoading best model from Google Drive...")

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])

print(f"Best epoch: {checkpoint['epoch']}")
print(f"Best validation accuracy: {checkpoint['val_accuracy']:.4f}")
print(f"Best validation Macro-F1: {checkpoint['val_macro_f1']:.4f}")

# ============================================================
# FINAL TEST
# ============================================================

model.eval()

all_predictions = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)

        logits = model(X_batch)

        predictions = (
            logits
            .argmax(dim=1)
            .cpu()
            .numpy()
        )

        all_predictions.extend(predictions)
        all_targets.extend(y_batch.numpy())

all_predictions = np.array(all_predictions)
all_targets = np.array(all_targets)

# ============================================================
# METRICS & RESULTS
# ============================================================

test_accuracy = accuracy_score(
    all_targets,
    all_predictions
)

test_macro_f1 = f1_score(
    all_targets,
    all_predictions,
    average="macro",
    zero_division=0
)

test_weighted_f1 = f1_score(
    all_targets,
    all_predictions,
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 60)
print("CNN #4 — TCN FINAL TEST RESULTS")
print("=" * 60)

print(f"Test Accuracy:    {test_accuracy:.4f}")
print(f"Test Macro-F1:    {test_macro_f1:.4f}")
print(f"Test Weighted-F1: {test_weighted_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        all_targets,
        all_predictions,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )  v
)

cm = confusion_matrix(
    all_targets,
    all_predictions
)

print("\nConfusion Matrix:")
print(cm)

per_class_f1 = f1_score(
    all_targets,
    all_predictions,
    average=None,
    zero_division=0
)

print("\nPer-Class F1:")
for label, score in zip(label_encoder.classes_, per_class_f1):
    print(f"{label}: {score:.4f}")

print("\n" + "=" * 60)
print("CNN #4 COMPLETE - MODEL STORED IN GOOGLE DRIVE")
print("=" * 60)

Raw X shape: (7594, 175)
Raw Y shape: (3797, 1)
Number of labels: 3797
Number of signal rows: 7594
Samples per signal: 175

Full dataset:
X: (3797, 2, 175)
y: (3797,)

After 25–75 ms extraction:
X: (3797, 2, 50)

Classes:
0 = Abnormal
1 = Artifact_high_gain
2 = Artifact_phase_shift
3 = Normal

Number of classes: 4

Splits:
Train: (2657, 2, 50) (2657,)
Val:   (570, 2, 50) (570,)
Test:  (570, 2, 50) (570,)

Device: cuda

Model:
TCN1D(
  (network): Sequential(
    (0): TemporalBlock(
      (conv1): Conv1d(2, 32, kernel_size=(3,), stride=(1,), padding=(2,))
      (chomp1): Chomp1d()
      (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu1): ReLU(inplace=True)
      (dropout1): Dropout(p=0.1, inplace=False)
      (conv2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(2,))
      (chomp2): Chomp1d()
      (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu2): ReLU(inplace=True)
      (drop